# 學生學習與作業版｜任務 2：檢查資料品質與日期

**姓名：**黃勉源 **學號／組別：**C114118246 **完成日期：**11/07

**學習目標：** 辨認缺失、重複、異常與含歧義的日期，保留原始紀錄。

本單元與教師版「任務 2」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「2.2 異常值」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [ ]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


模擬資料形狀（列, 欄）： (60, 7)


## 2.1 缺失與重複
**觀念說明：**缺失表示未知，不能當成 0。重複整列與識別碼重複是兩件事；先確認業務意義再決定是否刪除。
**預測：**睡眠欄位有幾筆缺失？其他欄位如何？


In [ ]:
# isna() 找缺失；sum() 在每欄加總 True 的數量
print(df.isna().sum())
print("整列完全重複：", df.duplicated().sum())
print("學生編號重複：", df["student_id"].duplicated().sum())
# 顯示含睡眠缺失的原始列，先觀察而不是直接刪除
display(df.loc[df["sleep_hours"].isna()])


student_id           0
department           0
sleep_hours          2
screen_time_hours    0
study_hours          0
attendance_rate      0
quiz_score           0
dtype: int64
整列完全重複： 0
學生編號重複： 0


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
5,6,金融管理,NaN,2.8,4.3,96.0,92.0
17,18,資訊管理,NaN,4.5,2.9,100.0,82.0


## 📝 學生作業 02-Q1｜AI 查詢與作答：未知與缺失
**任務：**「未知」、0、NaN 有什麼差異？未確認日期為何不能直接猜？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請比較未知、數值零、缺失值，並解釋 01-02-2026 在日月格式不明時應怎麼處理。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 02-Q1 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：使用高科大AI助理(kklc)，採用教材提供的提示詞

- **AI 回答重點（請自行整理）**：
  1. 數值零 (0)：代表明確的數值量度為「無」，會直接參與所有的數學運算（如計算平均值）。
  2. 缺失值 (NaN)：在 Pandas 中代表「漏失或未填寫的欄位」，計算平均或加總時預設會自動忽略，誤填成 0 會嚴重拉低統計平均。
  3. 未知 (Unknown)：通常是字串型態，代表業務邏輯上「無法確定」或資料損毀，不應直接與數值混合計算。
  4. 未確認日期（如 01-02-2026）：存在日月格式歧義（可以是 2 月 1 日或 1 月 2 日）。若直接猜測會導致時間序列、月報表等分析產生嚴重偏差。正確做法是保留原始字串，將其標記為需要人工查證（`needs_review = True`）。

- **我的理解與具體例子**：
  * 我的理解：如果班上有 3 個人的讀書時間分別是 2 小時、4 小時，第 3 個人沒填（應為 `NaN`）。如果直接算平均，正確應該是 $(2+4)/2 = 3$ 小時；但如果我們盲目地把 `NaN` 補成 `0`，平均值就會變成 $(2+4+0)/3 = 2$ 小時，這會嚴重誤導決策。
  * 驗證程式例：
    ```python
    import pandas as pd
    import numpy as np
    df_test = pd.DataFrame({'score': [80, 90, np.nan], 'score_filled_zero': [80, 90, 0]})
    print(df_test.mean()) # 可以觀察到兩者平均值的重大落差
    ```

- **查證來源／方式及日期（無法查證請註明）**：
  * 查證來源：Pandas 官方文件關於 `Working with missing data` 說明與 `pd.to_datetime` API 規範。
  * 查證日期：2026 年 11 月 7 日。

- **我檢查或修正 AI 回答的地方**：
  * 確認了 `NaN` 在 Pandas 中預設在 `mean()` 計算中會被跳過（`skipna=True`）。AI 所提的運算差異完全符合 Pandas 的預設行為，無須修正。

## 2.2 異常值需要依規格判斷








15 小時是在產生成績之後故意置入，超過本案例原始設定的 10 小時上限；真實生活中不能僅憑超過 10 小時就認定錯誤。先標記、查證與記錄。


## 💻 學生作業 02-P1｜Vibe Coding 程式填空
**對應教師版：**任務 2「2.2 異常值」的示範程式。

**任務：**建立布林篩選條件 flag，判斷 screen_time_hours 是否大於 10；請勿刪除或修改資料。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：建立布林篩選條件 flag，判斷 screen_time_hours 是否大於 10；請勿刪除或修改資料。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [ ]:
# 根據本案例原先生成規格標記高於 10 小時的紀錄
flag = df["screen_time_hours"] > 10  # TODO：找出本案例超過 10 小時的資料
print("超過案例生成上限的筆數：", flag.sum())
display(df.loc[flag, ["student_id", "screen_time_hours", "quiz_score"]])

超過案例生成上限的筆數： 1


,student_id,screen_time_hours,quiz_score
8,9,15.0,80.0


### ✅ 作業 02-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [ ]:
assert flag.equals(df["screen_time_hours"].gt(10))
assert len(flag) == len(df)
print("檢查通過：篩選條件正確。")


檢查通過：篩選條件正確。


### ✍️ 作業 02-P1｜我的完成紀錄
- 我實際使用的提示詞（若未使用 AI 請註明）：使用高科大AI助理(kklc)，採用教材提供的提示詞
- 我補上的程式在做什麼：對每一筆資料判斷「螢幕使用時間是否超過 10 小時」，符合為 True，不符合為 False，只增加標記欄位，不影響原有資料筆數。
- 執行或驗證遇到的問題與修正：要求超過10，因此為 > 而非 >=
- 檢查結果及我如何確認：檢查結果顯示共有 N 筆資料超過 10 小時上限，並列出其學號、時數與測驗分數；確認方式為先以 .gt(10) 重算一次並用 equals 比對，確認兩種寫法對每一列的判斷完全一致，再比對旗標長度與原資料列數，確認僅做標記、未刪減任何資料，兩項斷言皆通過。


## 📝 學生作業 02-Q2｜AI 查詢與作答：品質問題
**任務：**完整資料列重複與 student_id 重複有何不同？舉一個不能直接刪除的情境。

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請舉例比較完整資料列重複與學生編號重複，並說明為何需先確認資料記錄意義。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 02-Q2 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：使用高科大AI助理(kklc)，採用教材提供的提示詞
- AI 回答重點（請自行整理）：

  1.完整資料列重複：所有欄位值都完全相同，通常是同一筆紀錄被重複匯入或重複 concat，屬較單純的重複，刪除風險低。

  2.student_id 重複：學生編號相同，但至少有一個其他欄位不同。可能是輸入錯誤，也可能是資料設計本來就允許一位學生有多筆紀錄。

  3.兩者的關鍵差別在「一列資料代表什麼」（記錄粒度）：若一列代表「一位學生」，編號重複才是異常；若一列代表「一位學生的一天／一次測驗」，編號重複就是正常結構（長格式資料）。

  4.因此刪除前必須先確認資料的記錄意義，否則用 drop_duplicates(subset=["student_id"]) 會誤刪合法資料。

  5.建議檢查方式：df.duplicated().sum() 看完整列重複筆數；df[df.duplicated(subset=["student_id"], keep=False)] 把涉及編號重複的所有列都列出來比對。

- 我的理解與具體例子：完整資料列重複像「同一張表單被交兩次，內容一模一樣」；student_id 重複像「同一個人交了兩次，但兩次填的內容不同」，後者要先弄清楚是要保留最新一次、還是要全部保留。 不能直接刪除的情境：假設資料是為期兩週的每日追蹤，每位學生每天一列。

- 查證來源／方式及日期（無法查證請註明）：pandas 官方文件 DataFrame.duplicated 與 DataFrame.drop_duplicates 的參數說明（subset、keep），以及本次在 Python 環境實際執行上述小例子確認輸出；查證日期 2026-11-7。
- 我檢查或修正 AI 回答的地方：我把「完整列重複一定能刪」修正為「多數情況可刪，但仍建議先用 df[df.duplicated(keep=False)] 抽幾筆查看」，因為有可能某個事件真的發生了兩次且數值恰好相同。


## 2.3 日期格式與未知值：獨立銷售情境
**觀念說明：**`01-02-2026` 在日／月與月／日規則下可能是不同日期，不能猜。以下例子只轉換明確的 ISO 形式，以及日月數字相同、沒有日月歧義的 `01-01-2026`；「未知」、其他格式先保留為待查。

**提問：**`NaT` 是什麼？它是否表示 0 或可自行補成今天？


In [ ]:
# 四筆小資料專門用來示範日期品質，與學生資料主線分開
sales = pd.DataFrame({"sale_id": [1, 2, 3, 4],
                      "sale_date_raw": ["2026/01/01", "01-01-2026", "未知", "01-02-2026"],
                      "amount": [500, 800, 300, 600]})
sales_clean = sales.copy()  # 不改動原始資料
sales_clean["sale_date_clean"] = pd.NaT  # 新欄先設為無法確認
# 使用精確格式解析 YYYY/MM/DD；錯誤值標成 NaT，保留原值供追查
iso = sales_clean["sale_date_raw"].str.fullmatch(r"\d{4}/\d{2}/\d{2}")
sales_clean.loc[iso, "sale_date_clean"] = pd.to_datetime(
    sales_clean.loc[iso, "sale_date_raw"], format="%Y/%m/%d", errors="coerce")
# 範例中的 01-01-2026 無日月歧義；只處理這個已確認值
same_day_month = sales_clean["sale_date_raw"].eq("01-01-2026")
sales_clean.loc[same_day_month, "sale_date_clean"] = pd.to_datetime(
    sales_clean.loc[same_day_month, "sale_date_raw"], format="%d-%m-%Y", errors="coerce")
# 任何尚未確認的資料都保留原值，列入人工查證清單
sales_clean["needs_review"] = sales_clean["sale_date_clean"].isna()
display(sales_clean)
print("待查筆數：", sales_clean["needs_review"].sum())
assert sales_clean.loc[3, "needs_review"]  # 有歧義日期沒有被任意轉換


,sale_id,sale_date_raw,amount,sale_date_clean,needs_review
0,1,2026/01/01,500,2026-01-01,False
1,2,01-01-2026,800,2026-01-01,False
2,3,未知,300,NaT,True
3,4,01-02-2026,600,NaT,True


待查筆數： 2


## 延伸示範：彙整品質檢查紀錄
缺失、重複與超出範圍是不同問題，處理前應先有可追查的摘要。以下範圍上限只適用模擬資料生成規格。

**驗證提問：**資料筆數是否變少？為何 `sleep_hours` 的缺失不能直接補 0？日期未能轉換時，應保留原文，並洽資料來源確認規則。


In [ ]:
# 先彙整品質數量，不直接變更或刪除資料
quality_summary = pd.Series({
    "rows": len(df),
    "sleep_missing": int(df["sleep_hours"].isna().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_ids": int(df["student_id"].duplicated().sum()),
    "screen_above_10": int((df["screen_time_hours"] > 10).sum())})
display(quality_summary)
assert quality_summary["rows"] == 60


,0
rows,60
sleep_missing,2
duplicate_rows,0
duplicate_ids,0
screen_above_10,1


## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。
